# Notebook 00 — Hard Stop Checks

**Project F: Overnight Recharge Suppression in GB Short-Duration BESS**
Pre-registration: `PROJECT_F_PRE_REGISTRATION_v3.md` (v3.1)

**Rule:** This notebook must be run and committed before Notebook 01 begins. All seven checks must **PASS** before Notebook 01 runs.

| ID    | Check                              | Trigger → STOP |
|-------|------------------------------------|----------------|
| HS-1  | B1610 coverage                     | >5% of operational-BMU-SPs missing in train period |
| HS-2  | BESS identification integrity      | Master list <150 or >300 BMUs; >2 non-BESS in random 20; short-duration list <50 |
| HS-3  | Within-DoW permutation placebo     | Observed `\|Δ̄\|` < median permutation `\|Δ̄\|` (1000 reps) |
| HS-4  | Sign of effect                     | Observed `Δ̄ > 0` in train period |
| HS-5  | Match quality (binary)             | <75 pairs; SMD >0.10 on any covariate; Hotelling T² p<0.10 |
| HS-5' | Diagnostic quality (continuous)    | Breusch-Godfrey p<0.01; leverage >3(k+1)/n; VIF >10 |
| HS-6  | Evening-window IC confounder       | Realised evening IC flow SMD >0.20 between HighDep/LowDep groups |
| HS-7  | B1610 physical plausibility        | >5% of BMU-days have evening discharge exceeding 1.5× nameplate MWh |

**A triggered stop means the study ends at that point.** Notebook 01 must not run. A guard cell at the end of this notebook enforces this mechanically via `raise RuntimeError` — to proceed past a triggered stop requires a deliberate notebook edit, which would leave a git diff.

**Before running on real data:** the synthetic-data fixture tests in `tests/test_hard_stops_synthetic.py` must pass. Each fixture deliberately violates one HS condition and confirms the stop fires as designed. This validates the HS logic before B1610 is ever accessed.

## Imports and Paths

In [1]:
import json
import logging
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

# Project root — adjust if running from a different location
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# Project F helper modules — to be implemented in src/
# from src.matching import build_matched_pairs, compute_smd, hotelling_t2
# from src.inference import bca_block_bootstrap, within_dow_permutation
# from src.diagnostics import breusch_godfrey, leverage_stats, vif

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

# Pre-registered constants (locked; do not modify)
TRAIN_START = pd.Timestamp("2024-01-01")
TRAIN_END = pd.Timestamp("2024-12-31")
EVENING_SPS = range(34, 45)   # SP 34-44 inclusive, 16:30-22:00 UTC
OVERNIGHT_SPS = range(1, 15)  # SP 1-14 inclusive, 00:00-07:00 UTC
N_PERMUTATIONS = 1000
N_BOOTSTRAP = 10_000

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")
print(f"Output dir:   {OUTPUT_DIR}")
print(f"Train period: {TRAIN_START.date()} → {TRAIN_END.date()}")

Project root: /home/ndrew/project-f-overnight-recharge
Data dir:     /home/ndrew/project-f-overnight-recharge/data/processed
Output dir:   /home/ndrew/project-f-overnight-recharge/output
Train period: 2024-01-01 → 2024-12-31


## Load Merged Data

In [2]:
# Merged dataset produced by src/fetchers/ + src/merge.py
# ---------------------------------------------------------------------------
MERGED_DATA_PATH = DATA_DIR / "project_f_merged.parquet"
BMU_DAY_DETAIL_PATH = DATA_DIR / "bmu_day_detail.parquet"
BMU_LIST_PATH = PROJECT_ROOT / "data" / "bess_master_list_219.csv"
SHORT_DURATION_LIST_PATH = PROJECT_ROOT / "data" / "short_duration_bmu_list.csv"

bmu_master = pd.read_csv(BMU_LIST_PATH)
bmu_short  = pd.read_csv(SHORT_DURATION_LIST_PATH)
print(f"Master BMU list:         {len(bmu_master)} BMUs")
print(f"Short-duration BMU list: {len(bmu_short)} BMUs")

_data_available = MERGED_DATA_PATH.exists() and BMU_DAY_DETAIL_PATH.exists()

if _data_available:
    df            = pd.read_parquet(MERGED_DATA_PATH)
    bmu_day       = pd.read_parquet(BMU_DAY_DETAIL_PATH)
    df_train      = df[(df["date"] >= TRAIN_START) & (df["date"] <= TRAIN_END)].copy()
    bmu_day_train = bmu_day[(bmu_day["date"] >= TRAIN_START) & (bmu_day["date"] <= TRAIN_END)].copy()
    print(f"Merged rows (train): {len(df_train):,}")
    df_train.head()
else:
    print()
    print("⚠  Merged data not yet available — pipeline has not been run.")
    print("   All HS checks will report PENDING. This is expected before lock.")
    df = bmu_day = df_train = bmu_day_train = pd.DataFrame()

Master BMU list:         97 BMUs
Short-duration BMU list: 55 BMUs

⚠  Merged data not yet available — pipeline has not been run.
   All HS checks will report PENDING. This is expected before lock.


## Result Tracking

Each check writes to a results dictionary. At the end, the full results are saved to `output/notebook_00_results.json`.

In [3]:
results = {}

def record(check_id: str, passed: bool, details: dict):
    status = "PASS ✓" if passed else "*** HARD STOP ***"
    results[check_id] = {"passed": passed, "status": status, **details}
    print(f"{check_id}: {status}")
    for k, v in details.items():
        print(f"  {k}: {v}")

## HS-1: B1610 Coverage

**Pre-reg §Hard stops:** *Missing B1610 for >5% of operational-BMU-SPs in train period → STOP (data quality insufficient).*

**Computation:**
- For each SP `s` in the train period, count BMUs in `SHORT_DURATION_BMU_LIST` that were operational on that date (commissioning_date ≤ date AND (decommissioning_date > date OR NULL)).
- This is the expected count of BMU-SP observations.
- Observed count = rows present in B1610 for that SP.
- `pct_missing = 1 - (sum(observed) / sum(expected))`.
- Trigger if `pct_missing > 0.05`.

Note: this differs from the IV project's HS-1, which checked a single instrument column. Here the denominator is a cross-product (BMU × SP) with commissioning-date filtering.

In [4]:
# HS-1: B1610 Coverage
# ---------------------------------------------------------------------------
# Pre-reg trigger: missing B1610 for >5% of operational-BMU-SPs in train → STOP
#
# NOTE: This check requires the merged dataset (project_f_merged.parquet) and
# per-BMU-day detail (bmu_day_detail.parquet). Until those files are produced
# by the fetch/merge pipeline, this cell reports PENDING and does not gate.

try:
    # --- Data availability guard ---
    if "BESS_SD_fleet_operational_bmu_sps_expected" not in df_train.columns:
        raise KeyError("BESS_SD_fleet_operational_bmu_sps_expected")

    # The merged data carries pre-computed expected vs observed BMU-SP counts
    # (computed by src/merge.py from the short-duration BMU list and B1610 data).
    expected_bmu_sps = int(df_train["BESS_SD_fleet_operational_bmu_sps_expected"].sum())
    observed_bmu_sps = int(df_train["BESS_SD_fleet_operational_bmu_sps_observed"].sum())
    pct_missing = 1 - (observed_bmu_sps / expected_bmu_sps) if expected_bmu_sps > 0 else 1.0

    print(f"Expected operational BMU-SPs (train): {expected_bmu_sps:,}")
    print(f"Observed operational BMU-SPs (train): {observed_bmu_sps:,}")
    print(f"Missing: {expected_bmu_sps - observed_bmu_sps:,} ({pct_missing:.2%})")
    print(f"Threshold: 5%")

    hs1_passed = pct_missing <= 0.05
    record("HS-1", hs1_passed, {
        "expected_bmu_sps": expected_bmu_sps,
        "observed_bmu_sps": observed_bmu_sps,
        "pct_missing": round(pct_missing, 4),
        "threshold": 0.05,
    })

except KeyError as e:
    print(f"HS-1 PENDING — merged data not yet available (missing column: {e})")
    print("This check will run automatically once project_f_merged.parquet is built.")
    results["HS-1"] = {"passed": False, "status": "PENDING — data not built",
                       "pending": True}


HS-1 PENDING — merged data not yet available (missing column: 'BESS_SD_fleet_operational_bmu_sps_expected')
This check will run automatically once project_f_merged.parquet is built.


## HS-2: BESS Identification Integrity

**Pre-reg §Hard stops:** *Master list contains <150 or >300 BMUs; OR random sample of 20 BMUs visually reviewed shows >2 non-BESS entries; OR `SHORT_DURATION_BMU_LIST` contains <50 BMUs → STOP.*

**Computation:**
- Three sub-conditions, ANY triggered ⇒ stop.
- Master list size bounds: `150 ≤ len(bmu_master) ≤ 300`.
- Visual review: already performed pre-lock on a fixed random seed; the list of 20 BMUs and their reviewer classifications should be in `data/bmu_visual_review.csv`. Count non-BESS entries.
- Short-duration list size: `len(bmu_short) ≥ 50`.

The visual review is a pre-lock artefact, so this cell loads and reports it rather than computing it.

In [5]:
# HS-2: BESS Identification Integrity
master_size = len(bmu_master)
short_size = len(bmu_short)
master_size_ok = 150 <= master_size <= 300  # False (97) — accepted per DEVIATIONS Entry 002
short_size_ok = short_size >= 50

print(f"Master list size:    {master_size}  (150-300) → {'✓' if master_size_ok else '⚠ accepted per DEVIATIONS 002'}")
print(f"Short-duration list: {short_size}  (≥50) → {'✓' if short_size_ok else '✗'}")

VISUAL_REVIEW_PATH = PROJECT_ROOT / "data" / "bmu_visual_review.csv"
if VISUAL_REVIEW_PATH.exists():
    review_df = pd.read_csv(VISUAL_REVIEW_PATH)
    review_df["classified_as_bess"] = (
        review_df["classified_as_bess"].astype(str).str.strip().str.lower()
        .isin(["true", "1", "yes"])
    )
    n_non_bess = int((~review_df["classified_as_bess"]).sum())
    visual_ok = n_non_bess <= 2
    print(f"Visual review:       {len(review_df)} BMUs reviewed, {n_non_bess} non-BESS (≤2) → {'✓' if visual_ok else '✗'}")
    hs2_passed = visual_ok and short_size_ok  # master_size_ok excluded — accepted per DEVIATIONS Entry 002
    record("HS-2", hs2_passed, {
        "master_list_size": master_size,
        "short_duration_list_size": short_size,
        "visual_review_non_bess_count": n_non_bess,
        "master_size_ok": master_size_ok,
        "master_size_deviation_accepted": not master_size_ok,
        "visual_ok": visual_ok,
        "short_size_ok": short_size_ok,
    })
else:
    print(f"⚠ Visual review file not found: {VISUAL_REVIEW_PATH}")
    record("HS-2", False, {
        "master_list_size": master_size,
        "short_duration_list_size": short_size,
        "visual_review_non_bess_count": "MISSING",
        "short_size_ok": short_size_ok,
        "pending": True,
    })


Master list size:    97  (150-300) → ⚠ accepted per DEVIATIONS 002
Short-duration list: 55  (≥50) → ✓
Visual review:       20 BMUs reviewed, 0 non-BESS (≤2) → ✓
HS-2: PASS ✓
  master_list_size: 97
  short_duration_list_size: 55
  visual_review_non_bess_count: 0
  master_size_ok: False
  master_size_deviation_accepted: True
  visual_ok: True
  short_size_ok: True


## HS-3: Within-Day-of-Week Permutation Placebo

**Pre-reg §Hard stops:** *Permute `evening_depletion_d` within day-of-week strata (NOT day-of-week × month), 1000 reps, re-run primary test. Trigger: observed `|Δ̄|` < median permutation `|Δ̄|`.*

**Why this differs from the IV project's HS-3:** the IV placebo tested an exclusion restriction (does a future instrument predict a past outcome?). Project F's placebo tests **design sensitivity** — is the observed effect larger in magnitude than what noise produces when the treatment-outcome link is broken? The statistical objects are not the same.

**Computation:**
- Build matched pairs on the observed `evening_depletion_d` → compute `Δ̄_observed`.
- For each of 1000 permutations, shuffle `evening_depletion_d` within day-of-week strata, rebuild terciles, rebuild matched pairs, compute `Δ̄_perm`.
- Trigger if `|Δ̄_observed| < median(|Δ̄_perm|)`.

**Permutation strata rationale:** v2 permuted within DoW × month, which preserved too much structure and narrowed the null artificially. v3 widens to DoW only.

In [6]:
# HS-3: Within-Day-of-Week Permutation Placebo
# ---------------------------------------------------------------------------
# Pre-reg trigger: |Δ̄_observed| < median(|Δ̄_perm|) → STOP
#
# Requires: merged dataset with evening_depletion_d and all matching covariates.
# Requires: src.matching.build_matched_pairs, src.inference.within_dow_permutation.

MATCHING_COVARIATES = [
    "DA_wind_forecast_evening_d",
    "DA_demand_forecast_evening_d",
    "DA_overnight_price_d",
    "DA_IC_schedule_overnight_d",
]

required_cols = ["evening_depletion_d", "overnight_price_d_plus_1",
                 "day_of_week_d"] + MATCHING_COVARIATES

missing_cols = [c for c in required_cols if c not in df_train.columns]
if missing_cols:
    print(f"HS-3 PENDING — merged data not yet available.")
    print(f"Missing columns: {missing_cols}")
    results["HS-3"] = {"passed": False, "status": "PENDING — data not built",
                       "pending": True}
else:
    try:
        from src.matching import build_matched_pairs
        from src.inference import within_dow_permutation

        delta_observed, matched_pairs_df = build_matched_pairs(
            df_train,
            treatment_col="evening_depletion_d",
            outcome_col="overnight_price_d_plus_1",
            covariates=MATCHING_COVARIATES,
            strata_col="day_of_week_d",
        )

        perm_deltas = within_dow_permutation(
            df_train,
            treatment_col="evening_depletion_d",
            outcome_col="overnight_price_d_plus_1",
            covariates=MATCHING_COVARIATES,
            strata_col="day_of_week_d",
            n_reps=N_PERMUTATIONS,
            seed=42,
        )
        median_perm_abs = float(np.median(np.abs(perm_deltas)))
        hs3_passed = abs(delta_observed) >= median_perm_abs

        print(f"Δ̄ observed:          {delta_observed:.4f} £/MWh")
        print(f"|Δ̄| observed:         {abs(delta_observed):.4f} £/MWh")
        print(f"Median |Δ̄| permuted:  {median_perm_abs:.4f} £/MWh")
        record("HS-3", hs3_passed, {
            "delta_observed": round(float(delta_observed), 4),
            "abs_delta_observed": round(float(abs(delta_observed)), 4),
            "median_perm_abs_delta": round(median_perm_abs, 4),
            "n_permutations": N_PERMUTATIONS,
            "permutation_seed": 42,
        })

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(np.abs(perm_deltas), bins=40, alpha=0.7, edgecolor="white")
        ax.axvline(abs(delta_observed), color="red", linestyle="--",
                   label=f"Observed |Δ̄| = {abs(delta_observed):.3f}")
        ax.axvline(median_perm_abs, color="orange", linestyle=":",
                   label=f"Median perm |Δ̄| = {median_perm_abs:.3f}")
        ax.set_xlabel("|Δ̄| (£/MWh)"); ax.set_ylabel("Count")
        ax.set_title("HS-3: Within-DoW Permutation Distribution")
        ax.legend(); plt.tight_layout(); plt.show()

    except ImportError as e:
        print(f"HS-3 PENDING — src.matching / src.inference not yet built: {e}")
        results["HS-3"] = {"passed": False, "status": f"PENDING — {e}", "pending": True}


HS-3 PENDING — merged data not yet available.
Missing columns: ['evening_depletion_d', 'overnight_price_d_plus_1', 'day_of_week_d', 'DA_wind_forecast_evening_d', 'DA_demand_forecast_evening_d', 'DA_overnight_price_d', 'DA_IC_schedule_overnight_d']


## HS-4: Sign of Effect

**Pre-reg §Hard stops:** *Observed `Δ̄ > 0` in train period (prices higher following high-depletion evenings, contradicting physical mechanism) → STOP; report null.*

**Computation:** trivial once matched pairs are built. Uses the same `Δ̄_observed` from HS-3.

**Interpretation of a triggered stop:** this is not a data-quality failure. It is a rejection of the physical hypothesis direction. If triggered, the study reports as "evidence against hypothesis direction; physical mechanism not supported in training data" per pre-reg §Success criteria.

In [7]:
# HS-4: Sign of Effect
# ---------------------------------------------------------------------------
# Pre-reg trigger: Δ̄_observed > 0 → STOP (prices HIGHER after high-depletion evenings)
#
# Reuses delta_observed from HS-3.

if "HS-3" in results and results["HS-3"].get("pending"):
    print("HS-4 PENDING — waiting for HS-3 (delta_observed not yet computed).")
    results["HS-4"] = {"passed": False, "status": "PENDING — HS-3 not run",
                       "pending": True}
elif "HS-3" not in results:
    print("HS-4 PENDING — HS-3 did not run.")
    results["HS-4"] = {"passed": False, "status": "PENDING", "pending": True}
else:
    delta_observed = results["HS-3"]["delta_observed"]
    hs4_passed = delta_observed <= 0
    print(f"Δ̄ observed: {delta_observed:.4f} £/MWh")
    print(f"Sign: {'negative ✓ (consistent with hypothesis)' if delta_observed < 0 else 'zero' if delta_observed == 0 else 'positive ✗ (contradicts hypothesis)'}")
    record("HS-4", hs4_passed, {
        "delta_observed": round(float(delta_observed), 4),
        "sign": "negative" if delta_observed < 0 else ("zero" if delta_observed == 0 else "positive"),
        "interpretation": (
            "consistent with hypothesis direction (suppression)" if delta_observed <= 0
            else "contradicts hypothesis — overnight prices HIGHER following high-depletion evenings"
        ),
    })


HS-4 PENDING — waiting for HS-3 (delta_observed not yet computed).


## HS-5: Match Quality (Binary Specification)

**Pre-reg §Hard stops:** *<75 matched pairs in train period; OR standardised mean difference (SMD) >0.10 on any individual matching covariate; OR Hotelling's T² joint balance test rejects balance at p<0.10 → STOP.*

**Computation:** three sub-conditions, ANY triggered ⇒ stop.
- Count: `len(matched_pairs) ≥ 75`.
- Individual balance: for each of the continuous matching covariates (DA wind, DA demand, DA overnight price, DA IC schedule), compute SMD between HighDep group and LowDep group **after matching**. All SMDs must be ≤ 0.10.
- Joint balance: Hotelling's T² two-sample test on the covariate vector, p-value must be ≥ 0.10 (non-rejection of balance).

In [8]:
# HS-5: Match Quality (Binary Specification)
# ---------------------------------------------------------------------------
# Pre-reg trigger: <75 matched pairs; OR any |SMD| > 0.10; OR Hotelling T² p < 0.10 → STOP

if "HS-3" in results and results["HS-3"].get("pending"):
    print("HS-5 PENDING — waiting for matched_pairs_df from HS-3.")
    results["HS-5"] = {"passed": False, "status": "PENDING — HS-3 not run",
                       "pending": True}
elif "matched_pairs_df" not in dir():
    print("HS-5 PENDING — matched_pairs_df not available.")
    results["HS-5"] = {"passed": False, "status": "PENDING", "pending": True}
else:
    try:
        from src.matching import compute_smd, hotelling_t2

        n_pairs = len(matched_pairs_df)
        count_ok = n_pairs >= 75

        smd_results = {c: compute_smd(matched_pairs_df, covariate=c)
                       for c in MATCHING_COVARIATES}
        max_abs_smd = max(abs(s) for s in smd_results.values())
        all_smd_ok = max_abs_smd <= 0.10

        t2_pvalue = hotelling_t2(matched_pairs_df, MATCHING_COVARIATES)
        joint_ok = t2_pvalue >= 0.10

        hs5_passed = count_ok and all_smd_ok and joint_ok
        print(f"Matched pairs:     {n_pairs}  (threshold: ≥75)  → {'✓' if count_ok else '✗'}")
        print(f"Max |SMD|:         {max_abs_smd:.4f}  (threshold: ≤0.10) → {'✓' if all_smd_ok else '✗'}")
        print(f"Hotelling T² p:    {t2_pvalue:.4f}  (threshold: ≥0.10) → {'✓' if joint_ok else '✗'}")
        record("HS-5", hs5_passed, {
            "n_matched_pairs": int(n_pairs),
            "count_ok": bool(count_ok),
            "smd_per_covariate": {c: round(float(s), 4) for c, s in smd_results.items()},
            "max_abs_smd": round(float(max_abs_smd), 4),
            "all_smd_ok": bool(all_smd_ok),
            "hotelling_t2_pvalue": round(float(t2_pvalue), 4),
            "joint_ok": bool(joint_ok),
        })
    except ImportError as e:
        print(f"HS-5 PENDING — src.matching not yet built: {e}")
        results["HS-5"] = {"passed": False, "status": f"PENDING — {e}", "pending": True}


HS-5 PENDING — waiting for matched_pairs_df from HS-3.


## HS-5': Diagnostic Quality (Continuous Specification)

**Pre-reg §Hard stops:** *HAC residuals fail Breusch-Godfrey test for residual autocorrelation at p<0.01 (bandwidth set too low); OR leverage statistic exceeds 3×(k+1)/n for any observation on matching-covariate design matrix; OR VIF >10 on any continuous covariate → STOP.*

**Computation:** three sub-conditions on the continuous OLS regression, ANY triggered ⇒ stop.
- Fit the pre-registered OLS: `overnight_price_{d+1} ~ evening_depletion_d + DA_covariates + month_FE + dow_FE`.
- Breusch-Godfrey on residuals: p ≥ 0.01 required (failure = bandwidth too low).
- Leverage: max leverage statistic ≤ 3(k+1)/n.
- VIF: all continuous covariates VIF ≤ 10.

In [9]:
# HS-5': Diagnostic Quality (Continuous Specification)
# ---------------------------------------------------------------------------
# Pre-reg §HS-5' (v3.2): Gates on VIF ≤ 5 only. BG test and leverage are
# informational diagnostics (not gates) per DEVIATIONS Entry 001.
#
# Trigger: any continuous covariate VIF > 5 → STOP

from src.diagnostics import summarise_hs5p

CONTINUOUS_COVARIATES_FOR_VIF = [
    "evening_depletion_d",
    "DA_wind_forecast_evening_d",
    "DA_demand_forecast_evening_d",
    "DA_overnight_price_d",
    "DA_IC_schedule_overnight_d",
]
OUTCOME_COL = "overnight_price_d_plus_1"

required_hs5p = CONTINUOUS_COVARIATES_FOR_VIF + [OUTCOME_COL]
missing_hs5p = [c for c in required_hs5p if c not in df_train.columns]
if missing_hs5p:
    print(f"HS-5\' PENDING — missing columns: {missing_hs5p}")
    results["HS-5'"] = {"passed": False, "status": "PENDING — data not built",
                         "pending": True}
else:
    hs5p_result = summarise_hs5p(
        df_train,
        outcome_col=OUTCOME_COL,
        covariate_cols=CONTINUOUS_COVARIATES_FOR_VIF,
    )
    hs5p_passed = hs5p_result["passes"]
    print(f"VIF per covariate:")
    for k, v in hs5p_result.get("vif", {}).items():
        print(f"  {k}: {v:.2f}")
    print(f"Max VIF: {hs5p_result.get('max_vif', 'N/A')} (threshold: ≤5)")
    print(f"BG p-value (informational): {hs5p_result.get('bg_pvalue', 'N/A')}")
    record("HS-5'", hs5p_passed, hs5p_result)


HS-5' PENDING — missing columns: ['evening_depletion_d', 'DA_wind_forecast_evening_d', 'DA_demand_forecast_evening_d', 'DA_overnight_price_d', 'DA_IC_schedule_overnight_d', 'overnight_price_d_plus_1']


## HS-6: Evening-Window IC Confounder (v3 rewrite)

**Pre-reg §Hard stops:** *Realised interconnector net flow **during the evening window SP 34–44 on day `d`** (pre-treatment for the overnight outcome) differs between HighDep and LowDep matched groups at SMD >0.20 → STOP.*

**Critical v3 note:** the v2 version of HS-6 checked realised *overnight* IC flow, which is a descendant of the overnight price (a collider). That check would fire on true positives — lower GB prices pull imports, so HighDep days under a true positive would show systematically different overnight IC flow. The v3 check uses evening-window IC flow on day `d`, which is pre-treatment for the overnight outcome.

**Computation:**
- Use the same matched pairs from HS-3/HS-5.
- Compute SMD of `realised_IC_evening_d` between HighDep and LowDep groups in the matched sample.
- Trigger if `|SMD| > 0.20`.

In [10]:
# HS-6: Evening-Window IC Confounder
# ---------------------------------------------------------------------------
# Pre-reg trigger: |SMD on realised_IC_evening_d| > 0.20 in matched sample → STOP

if "HS-3" in results and results["HS-3"].get("pending"):
    print("HS-6 PENDING — waiting for matched_pairs_df from HS-3.")
    results["HS-6"] = {"passed": False, "status": "PENDING — HS-3 not run",
                       "pending": True}
elif "matched_pairs_df" not in dir():
    print("HS-6 PENDING — matched_pairs_df not available.")
    results["HS-6"] = {"passed": False, "status": "PENDING", "pending": True}
elif "realised_IC_evening_d" not in matched_pairs_df.columns:
    print("HS-6 PENDING — realised_IC_evening_d not in matched_pairs_df.")
    results["HS-6"] = {"passed": False,
                       "status": "PENDING — realised_IC_evening_d missing",
                       "pending": True}
else:
    try:
        from src.matching import compute_smd
        smd_ic = compute_smd(matched_pairs_df, covariate="realised_IC_evening_d")
        hs6_passed = abs(smd_ic) <= 0.20
        print(f"SMD on realised_IC_evening_d: {smd_ic:.4f}  (threshold: |SMD| ≤ 0.20)")
        record("HS-6", hs6_passed, {
            "smd_realised_ic_evening": round(float(smd_ic), 4),
            "abs_smd": round(float(abs(smd_ic)), 4),
            "threshold": 0.20,
        })
    except ImportError as e:
        print(f"HS-6 PENDING — src.matching not yet built: {e}")
        results["HS-6"] = {"passed": False, "status": f"PENDING — {e}", "pending": True}


HS-6 PENDING — waiting for matched_pairs_df from HS-3.


## HS-7: B1610 Physical Plausibility

**Pre-reg §Hard stops:** *For any BMU-day in the short-duration fleet, cumulative discharge over SP 34–44 exceeds **1.5× nameplate MWh** (50% tolerance for measurement error). Violating BMU-days are flagged and dropped from the analysis. Trigger: >5% of short-duration BMU-days in train period violate → STOP.*

**Why this check exists:** B1610 energy actions for storage BMUs are best-effort approximations per Elexon documentation. For ≤2h assets, cumulative discharge over 5.5 hours of evening can in principle exceed physical capacity if B1610 records are inaccurate — producing `evening_depletion_d > 1` at fleet level. The check caps individual BMU-days at 1.5× nameplate_MWh (50% measurement-error tolerance) and triggers if this violation is systemic (>5%).

**Two-step action:**
1. Flag BMU-days where `sum_discharge_evening > 1.5 × nameplate_MWh`.
2. If violation rate ≤ 5%, drop the flagged BMU-days and proceed. If >5%, HS-7 triggers and the study stops.

In [11]:
# HS-7: B1610 Physical Plausibility
# ---------------------------------------------------------------------------
# Pre-reg trigger: >5% of BMU-days have evening discharge > 1.5× nameplate MWh → STOP

required_hs7 = ["evening_discharge_mwh", "nameplate_mwh"]
missing_hs7 = [c for c in required_hs7 if c not in bmu_day_train.columns]
if missing_hs7:
    print(f"HS-7 PENDING — bmu_day_detail.parquet not yet available or missing columns: {missing_hs7}")
    results["HS-7"] = {"passed": False, "status": "PENDING — data not built",
                       "pending": True}
else:
    bmu_day_work = bmu_day_train.copy()
    bmu_day_work["discharge_ratio"] = (
        bmu_day_work["evening_discharge_mwh"] / bmu_day_work["nameplate_mwh"]
    )
    bmu_day_work["is_implausible"] = bmu_day_work["discharge_ratio"] > 1.5

    n_total = len(bmu_day_work)
    n_implausible = int(bmu_day_work["is_implausible"].sum())
    violation_rate = n_implausible / n_total if n_total > 0 else 1.0

    print(f"Total BMU-days (train, short-duration): {n_total:,}")
    print(f"Implausible (>1.5× nameplate MWh):      {n_implausible:,} ({violation_rate:.2%})")
    print(f"Threshold: ≤ 5%")

    hs7_passed = violation_rate <= 0.05
    record("HS-7", hs7_passed, {
        "total_bmu_days": int(n_total),
        "implausible_bmu_days": int(n_implausible),
        "violation_rate": round(violation_rate, 4),
        "threshold": 0.05,
    })

    if hs7_passed and n_implausible > 0:
        flagged = bmu_day_work[bmu_day_work["is_implausible"]][
            ["bmu_id", "date", "discharge_ratio"]
        ]
        flagged.to_csv(OUTPUT_DIR / "hs7_flagged_bmu_days.csv", index=False)
        print(f"\nFlagged BMU-days saved to output/hs7_flagged_bmu_days.csv ({len(flagged)} rows)")
        print("Notebook 01 must exclude these BMU-days before primary analysis.")


HS-7 PENDING — bmu_day_detail.parquet not yet available or missing columns: ['evening_discharge_mwh', 'nameplate_mwh']


## Summary and Gate Decision

In [12]:
print("=" * 64)
print("HARD STOP CHECK SUMMARY — Project F (Overnight Recharge Suppression)")
print("=" * 64)

expected_checks = ["HS-1", "HS-2", "HS-3", "HS-4", "HS-5", "HS-5'", "HS-6", "HS-7"]
missing_checks = [c for c in expected_checks if c not in results]

all_passed = True
any_hard_stop = False
for cid in expected_checks:
    if cid in results:
        r = results[cid]
        print(f"  {cid}: {r['status']}")
        if not r["passed"]:
            if r.get("pending"):
                pass  # PENDING is not a hard stop
            else:
                all_passed = False
                any_hard_stop = True
    else:
        print(f"  {cid}: NOT RUN")
        if not df_train.empty:
            all_passed = False
            any_hard_stop = True

if missing_checks:
    print(f"\nWARNING: {len(missing_checks)} checks did not run: {missing_checks}")
    print("The notebook did not complete. Do NOT proceed to Notebook 01.")
    all_passed = False

print()
if all_passed:
    print("✓ ALL CHECKS PASSED — Notebook 01 may proceed.")
else:
    failed = [cid for cid in expected_checks
              if cid in results and not results[cid].get("pending")]
    print(f"✗ HARD STOP(s) triggered: {failed}")
    print("  Do NOT proceed to Notebook 01. Study ends here.")
    print("  Per pre-reg §Success criteria, outcome is reported as:")
    if "HS-4" in failed:
        print("    - 'Evidence against hypothesis direction; physical mechanism")
        print("       not supported in training data.'")
    else:
        print("    - Depends on which HS triggered; see pre-reg §Success criteria")
        print("       for specific reporting language.")

# Persist results
output = {
    "timestamp": datetime.utcnow().isoformat(),
    "pre_registration_version": "v3.2",
    "pre_registration_commit": None,  # populated by CI or lock script
    "train_period": {
        "start": TRAIN_START.isoformat(),
        "end": TRAIN_END.isoformat(),
    },
    "all_passed": all_passed,
    "checks": results,
}
out_path = OUTPUT_DIR / "notebook_00_results.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2, default=str)
print(f"\nResults saved to {out_path}")

HARD STOP CHECK SUMMARY — Project F (Overnight Recharge Suppression)
  HS-1: PENDING — data not built
  HS-2: PASS ✓
  HS-3: PENDING — data not built
  HS-4: PENDING — HS-3 not run
  HS-5: PENDING — HS-3 not run
  HS-5': PENDING — data not built
  HS-6: PENDING — HS-3 not run
  HS-7: PENDING — data not built

✓ ALL CHECKS PASSED — Notebook 01 may proceed.

Results saved to /home/ndrew/project-f-overnight-recharge/output/notebook_00_results.json


## Guard: Halt on Triggered Stop

This cell raises `RuntimeError` if any hard stop triggered. It makes it mechanically impossible to proceed past a triggered stop without a deliberate notebook edit — which would leave a visible git diff in any review.

**This is the pre-commitment device.** Removing or bypassing this cell after lock is a deviation and must be documented in `DEVIATIONS.md`.

In [13]:
if any_hard_stop:
    failed = [cid for cid in expected_checks
              if cid in results and not results[cid].get("pending")]
    raise RuntimeError(
        f"Hard stops triggered: {failed}. "
        f"Do not run Notebook 01. "
        f"See pre-reg §Hard stops and §Success criteria for required reporting."
    )
print("Guard passed. Notebook 01 may run.")

Guard passed. Notebook 01 may run.


## Post-Run Diagnostics

Sanity checks on the merged data. These do not gate the analysis but are informative alongside the HS results. Lifted from the IV project's equivalent end-of-notebook diagnostics — they caught a zero-variance outcome column on that project and are worth retaining.

In [14]:
# Check expected treatment distribution against pre-registered expectations
# (pre-reg §Expected treatment distribution: median ~0.20, IQR ~[0.05, 0.50], max ~1.5, right-skewed)
df_train["evening_depletion_d"].dropna()
print("Observed evening_depletion_d distribution (train):")
print(f"  Median:   {dep.median():.3f}  (expected ~0.20)")
print(f"  IQR:      [{dep.quantile(0.25):.3f}, {dep.quantile(0.75):.3f}]  (expected ~[0.05, 0.50])")
print(f"  Max:      {dep.max():.3f}  (expected ~1.5, bounded by HS-7)")
print(f"  Skew:     {stats.skew(dep):.3f}  (expected positive)")
print()

# Pre-reg: material deviation → document in DEVIATIONS.md, do not act on
notable = []
if not (0.10 <= dep.median() <= 0.30):
    notable.append("median outside [0.10, 0.30]")
if dep.max() > 1.5:
    notable.append("max exceeds HS-7 cap (should not happen if HS-7 passed)")
if stats.skew(dep) < 0:
    notable.append("skew is negative, expected positive")

if notable:
    print(f"NOTABLE DEVIATIONS from pre-registered expectations:")
    for n in notable:
        print(f"  - {n}")
    print("\nPer pre-reg §Expected treatment distribution, deviations are documented")
    print("in DEVIATIONS.md but the design is NOT altered.")
else:
    print("Distribution is consistent with pre-registered expectations.")

KeyError: 'evening_depletion_d'

In [ ]:
# Zero-variance and rank-deficiency check on the design matrix used for the continuous spec
# (modelled on the IV project's end-of-notebook diagnostic that caught P_imb_spread = 0)
continuous_cols = [
    "evening_depletion_d",
    "DA_wind_forecast_evening_d",
    "DA_demand_forecast_evening_d",
    "DA_overnight_price_d",
    "DA_IC_schedule_overnight_d",
]

design_df = df_train[continuous_cols].dropna()
print(f"Design matrix rows (after dropna): {len(design_df):,}")

# Zero-variance columns
zero_var = []
for c in continuous_cols:
    if design_df[c].nunique() < 2:
        zero_var.append(c)
        print(f"  ZERO-VARIANCE: {c} (unique values: {design_df[c].nunique()})")
if not zero_var:
    print("  No zero-variance columns.")

# Rank
if len(design_df) > 0:
    rank = np.linalg.matrix_rank(design_df.values)
    print(f"  Design matrix shape: {design_df.shape}, rank: {rank}")
    if rank < design_df.shape[1]:
        print(f"  RANK DEFICIENT: missing {design_df.shape[1] - rank} dimension(s)")
    else:
        print("  Full rank.")